<a href="https://colab.research.google.com/github/ksealmalaysia41-lang/.github/blob/master/Stripe_Issuing_Decline_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import json
import time

# Configuration
API_KEY = "sk_test_yourSecretKey" # Replace with your test key
CARD_ID = "ic_123456789"         # Replace with your test card ID
BASE_URL = "https://api.stripe.com/v1"

def update_card_limit(limit_amount_usd):
    """
    Sets a monthly spending limit on the card.
    Note: Stripe API uses cents (100000 = $1,000.00).
    """
    url = f"{BASE_URL}/issuing/cards/{CARD_ID}"
    payload = {
        "spending_controls[spending_limits][0][amount]": limit_amount_usd * 100,
        "spending_controls[spending_limits][0][currency]": "usd",
        "spending_controls[spending_limits][0][interval]": "monthly"
    }

    response = requests.post(url, auth=(API_KEY, ""), data=payload)
    if response.status_code == 200:
        print(f"✅ Successfully set monthly limit to ${limit_amount_usd} USD")
    else:
        print(f"❌ Failed to update card: {response.text}")

def simulate_authorization(amount_usd, merchant_name="REDEYE Security"):
    """
    Simulates a card authorization attempt.
    """
    url = f"{BASE_URL}/issuing/authorizations"
    payload = {
        "amount": amount_usd * 100,
        "currency": "usd",
        "card": CARD_ID,
        "merchant_data[name]": merchant_name
    }

    print(f"🚀 Attempting ${amount_usd} authorization at {merchant_name}...")
    response = requests.post(url, auth=(API_KEY, ""), data=payload)
    data = response.json()

    if data.get("approved"):
        print(f"🟢 APPROVED: Auth ID {data.get('id')}")
    else:
        reason = data.get("request_history", [{}])[0].get("reason", "unknown")
        print(f"🔴 DECLINED: Reason - {reason}")

    return data

if __name__ == "__main__":
    # 1. Set the limit to $1,000 as requested
    update_card_limit(1000)

    # 2. Test a successful transaction ($500)
    simulate_authorization(500)

    # 3. Test the decline ($501) - Total ($1001) exceeds $1000 limit
    # This matches the JSON output you provided in the chat.
    simulate_authorization(501)

❌ Failed to update card: {
  "error": {
    "message": "Invalid API Key provided: sk_test_*********tKey",
    "type": "invalid_request_error"
  }
}

🚀 Attempting $500 authorization at REDEYE Security...
🔴 DECLINED: Reason - unknown
🚀 Attempting $501 authorization at REDEYE Security...
🔴 DECLINED: Reason - unknown
